# GGIS 407 - Lab 3: Geospatial Object Manipulation Using Python Libraries
#### Interactive Jupyter Notebook

This notebook provides an introduction to manipulation of <b>geospatial data</b> using Python. Both vector data manipulation using <b>Shapely</b> and raster data manipulation using <b>Rasterio</b> are covered.


## Notebook Outline: 

- [Introducing Shapely](#shapely_intro)
    - [Basic Shapes](#basic_shapes)
    - [Geometric Attributes and Methods](#geom_ops)
    - [Evaluate Topological Relationships](#top_rel)
- [Exploring Rasterio](#ras_io)

<a id='shapely_intro'></a>
## Introducing Shapely

<b>Shapely</b> is a really helpful library for working with geometries. You can use it both for simple geometric operations and working with complicated real-world geometries.

<a id='basic_shapes'></a>
### Basic Shapes

Some useful shapes are `Point`s, `Line`s, and `Polygon`s. You can combine them into `MultiPoint`s, `MultiLineString`s, and `MultiPolygon`s. Here are a few examples of creating and viewing basic shapes:

In [ ]:
import shapely
import shapely.geometry

In [ ]:
# create point

sh_point = shapely.geometry.Point(0, 1)

In [ ]:
# view point

sh_point

In [ ]:
# view object type

type(sh_point)

In [ ]:
# create line

sh_line = shapely.geometry.LineString([(0, 0), (1, 1)])
sh_line2 =  shapely.geometry.LineString([(0, 1), (1, 1)])

In [ ]:
def line_intersect(l1,l2): 
    return l1.intersects(l2)

line_intersect(sh_line,sh_line2)

In [ ]:
# view line

sh_line

In [ ]:
# view object type

type(sh_line)

In [ ]:
sh_line.length

In [ ]:
# create polygons

sh_polygon1 = shapely.geometry.Polygon([(0, 0), (1, 1), (1, 0)])
sh_polygon2 = shapely.geometry.Polygon([(2, 2), (3, 3), (3, 2), (2.75, 2.25)])

In [ ]:
# view polygon

sh_polygon1

In [ ]:
# view object type

type(sh_polygon1)

In [ ]:
# create multipolygons

sh_polygons = shapely.geometry.MultiPolygon([sh_polygon1, sh_polygon2])

In [ ]:
# view multipolygons

sh_polygons

In [ ]:
# view object type

type(sh_polygons)

Note that the length is two - there are two polygons! The individual polygons in a multi-part shape are in its <b>`geoms`</b> attribute.

In [ ]:
# view length

len(sh_polygons.geoms)

Well Known Text, or <b>`wkt`</b> is a human-readable way to represent geometry.

In [ ]:
# view text

sh_polygons.wkt

In [ ]:
sh_polygons.wkb

<a id='geom_ops'></a>
### Geometric Attributes and Methods

Once you have constructed your shapes, there are a variety of attributes you can look up or methods you can use on the geometric objects. Here are a few examples:

In [ ]:
# find centroid

sh_polygons.centroid

In [ ]:
# find centroid text

sh_polygons.centroid.wkt

In [ ]:
# find area

sh_polygons.area

In [ ]:
# find length

sh_polygons.length

In [ ]:
# find boundary

sh_polygons.boundary

In [ ]:
# find bounds

sh_polygons.bounds

In [ ]:
# find envelope

sh_polygons.envelope

A polygon is created from a <b>set of rings</b> representing <b>interior</b> and <b>exterior</b> coordinates.

In [ ]:
# create polygon with hole

sh_polygon3 = shapely.geometry.Polygon([(0, 0), (0, 1), (1, 1), (1, 0)], 
                                       [[(0.25, 0.25), (0.75, 0.25), (0.75, 0.75)],
                                       [(0.25, 0.5), (0.25, 0.75), (0.5, 0.75)]])

In [ ]:
# view polygon

sh_polygon3

In [ ]:
# view exterior coordinates

list(sh_polygon3.exterior.coords)

In [ ]:
# view interior coordinates

for inter in list(sh_polygon3.interiors):
    print(inter.bounds)

<a id='top_rel'></a>
### Evaluate Topological Relationships

In [ ]:
coords = [(0, 0), (1, 1)]
shapely.geometry.LineString(coords).contains(shapely.geometry.Point(0.5, 0.5))

In [ ]:
shapely.geometry.Point(0.5, 0.5).within(shapely.geometry.LineString(coords))

In [ ]:
#Endpoint of boundry not contained in line 
shapely.geometry.LineString(coords).contains(shapely.geometry.Point(1.0, 1.0)) 

In [ ]:
a = shapely.geometry.LineString([(0, 0), (1, 1)])
b = shapely.geometry.LineString([(1, 1), (2, 2)])

In [ ]:
# Returns True if the objects have at least one point in common 
# and their interiors do not intersect with any part of the other.
# Overlapping features do not therefore touch
a.touches(b)

In [ ]:
# Returns True if the boundary or interior of the object 
# intersect in any way with those of the other 
a.intersects(b)

In [ ]:
# Returns True if the objects intersect 
# but neither contains the other.
a.overlaps(b)

In [ ]:
# Returns True if the object’s boundary and interior 
# intersect only with the interior of the other 
# (not its boundary or exterior).
a.within(b)

In [ ]:
# Returns True if no points of other lie in the 
# exterior of the object and at least one point of the 
# interior of other lies in the interior of object.
a.contains(b)

In [ ]:
# Practice what you have learned about Shapely
# Feel free to modilfy any code above

<a id='ras_io'></a>
## Exploring Rasterio

In [ ]:
import rasterio

In [ ]:
# Download the USDA NASS Cropland Data Layer (2024) for Champaign County, IL (FIPS 17019)
# from CropScape into the data folder. It is skipped if the file is already there.
import os, re, urllib.request

os.makedirs('data', exist_ok=True)
cdl_path = 'data/CDL_2024_Champaign.tif'

if not os.path.exists(cdl_path):
    api = 'https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile?year=2024&fips=17019'
    reply = urllib.request.urlopen(api, timeout=120).read().decode()
    tif_url = re.search(r'<returnURL>(.*?)</returnURL>', reply).group(1)   # CropScape replies with a link to the GeoTIFF
    urllib.request.urlretrieve(tif_url, cdl_path)

print('Raster ready:', cdl_path, round(os.path.getsize(cdl_path) / 1e6, 1), 'MB')

In [ ]:
# Rasterio's rasterio.open() takes a path and returns a dataset object.
# The path may point to a file of any supported raster format.
# Rasterio will open it using the proper GDAL format driver.
# Dataset objects have some of the same attributes as Python file objects.
#
# Data source: USDA NASS Cropland Data Layer 2024, Champaign County, via CropScape
# https://nassgeodata.gmu.edu/CropScape/

IL_ras = rasterio.open(cdl_path)

In [ ]:
IL_ras.name

In [ ]:
IL_ras.mode

In [ ]:
IL_ras.closed

In [ ]:
IL_ras.height

In [ ]:
IL_ras.width

In [ ]:
IL_ras.bounds

In [ ]:
IL_ras.crs

In [ ]:
IL_ras.indexes

In [ ]:
# The datatype of the bands 
(IL_ras.indexes,IL_ras.dtypes)

In [ ]:
band1=IL_ras.read(1)

In [ ]:
band1

### Rasterio reads raster data into numpy arrays so plotting a single band as two dimensional data can be accomplished directly with pyplot.

In [ ]:
from matplotlib import pyplot as plt
plt.imshow(IL_ras.read(1))

### Rasterio also provides rasterio.plot.show to perform common tasks such as displaying multi-band images as RGB and labeling the axes with proper geo-referenced extents.

In [ ]:
from rasterio.plot import show
show(IL_ras)

### Show some metadata for the raster

In [ ]:
IL_ras.meta

In [ ]:
# What is that EPSG code? Look it up with pyproj
from pyproj import CRS

#crs = CRS.from_user_input(IL_ras.crs)

crs = CRS.from_epsg(IL_ras.crs.to_epsg())
print("EPSG code:", crs.to_epsg())
print("Projected coordinate system is: ", crs.name)
print("Geographic coordinate system is: ", crs.geodetic_crs.name)

In [ ]:
# Practice what you have learned about Rasterio
# Feel free to modify any code above
